# 05 — Scientific Machine Learning & Baselines

**CuSelect-AI · Campaign `CU-001`**  
**Project:** Evidence-grounded multi-agent design of Cu–Ag catalyst hypotheses for CO₂ electroreduction  
**Input:** `data/processed/04_structures/` (Notebook 04's eight exported, unrelaxed atomic structures)  
**Scope:** Structural descriptors, interpretable non-ML baselines, conditional interatomic-potential diagnostics, and honest readiness assessment.  
**Status at entry:** 8/8 geometrically validated; **0 human-approved for production modeling; 0 trustworthy energy/selectivity labels for these structures**.

### Central scientific question

> Can we construct a reproducible, scientifically constrained computational baseline for Cu–Ag atomic structures and assess whether a pretrained interatomic potential is appropriate, **without confusing structural differences or model-estimated energies with electrochemical selectivity**?

### Explicit non-goals

- No ethylene/ethanol Faradaic-efficiency prediction or ML regression from three verified claims in a single paper.
- No assertion that a candidate is thermodynamically stable based solely on unrelaxed geometry.
- No promotion of published P005 Cu(111) DFT source values to CuSelect-AI structure-specific labels.
- No training of a selectivity model or benchmarking against the same data it was trained on.
- **No file writes or pretrained-model downloads by default.**

This notebook is built to run on the `cuselect` Python 3.11 environment on Apple Silicon. It uses CPU-first operations; larger MLIP experiments require a separate resource and applicability review.

## 5.1 — Reproducibility contract and source inventory

The authoritative atomic geometries are the exported files from Notebook 04, not any lingering Jupyter variables. Confirm the registry and manifest are internally consistent before calculating descriptors or enabling any calculator. Structural provenance is a prerequisite, but not a modeling approval.

In [1]:
# PSEUDOCODE
# 1. Locate the project root and the exported Notebook 04 library.
# 2. Load the structure registry and manifest without modifying them.
# 3. Check the eight-file inventory and inherited scientific restrictions.
# 4. Print the exact library location and package versions.

from pathlib import Path
import hashlib
import importlib.metadata as metadata
import importlib.util
import json
import platform
import numpy as np
import pandas as pd
from IPython.display import display
import ase
from ase.io import read

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").is_file()), None)
assert ROOT is not None, "Run this notebook within the CuSelect-AI project."
EXPORT_ROOT = ROOT / "data" / "processed" / "04_structures"
assert EXPORT_ROOT.is_dir(), "Notebook 04 structure export is missing."
REGISTRY_PATH = EXPORT_ROOT / "04_structure_registry.csv"
MANIFEST_PATH = EXPORT_ROOT / "04_structure_manifest.json"
assert REGISTRY_PATH.is_file() and MANIFEST_PATH.is_file()

registry05 = pd.read_csv(REGISTRY_PATH, dtype=str, keep_default_na=False)
manifest05 = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
structure_paths = sorted(EXPORT_ROOT.glob("*.extxyz"))
assert len(registry05) == len(structure_paths) == manifest05["candidate_count"] == 8
assert registry05["candidate_id"].is_unique
assert not registry05["approved_for_modeling"].str.strip().str.lower().eq("true").any()
assert registry05["structure_review_status"].eq("pending_visual_review").all()
assert registry05["geometry_status"].eq("unrelaxed").all()
assert manifest05["ml_approved"] is False
assert set(manifest05["files"]) == {p.name for p in structure_paths}
assert set(registry05["candidate_id"]) == {p.stem for p in structure_paths}

print("NOTEBOOK 05 — REPRODUCIBILITY CONTRACT: PASSED")
print("Project root:", ROOT)
print("Python:", platform.python_version(), "| ASE:", ase.__version__, "| NumPy:", np.__version__)
print("Source geometries:", len(structure_paths))
print("Human visual review:", manifest05["human_review_status"])
print("Modeling approved:", manifest05["ml_approved"])
print("No input files modified; no models downloaded")

NOTEBOOK 05 — REPRODUCIBILITY CONTRACT: PASSED
Project root: /Users/nsolebna/202609_Ironhack/Week 8/Project 3/CuSelect-AI
Python: 3.11.18 | ASE: 3.29.0 | NumPy: 2.4.6
Source geometries: 8
Human visual review: pending_visual_review
Modeling approved: False
No input files modified; no models downloaded


## 5.2 — Independent structure reload and full-geometry fingerprints

Notebook 04 committed a SHA-256 fingerprint of **atom ordering, atomic numbers, positions, layer tags, cell vectors, and periodic boundary conditions**, rounded consistently for serialization. Recreate the same fingerprint function independently and verify all eight exported structures.

This is a read-only provenance test. A checksum match verifies geometry identity within the defined serialization convention, **not** the physical correctness of the model.

In [2]:
# PSEUDOCODE
# 1. Recreate Notebook 04's canonical geometry fingerprint.
# 2. Reload every EXTXYZ geometry from disk.
# 3. Compare fingerprints and structural metadata to the registry.
# 4. Reject duplicate, corrupted, or inconsistent structures.
# 5. Produce a read-only structure audit table.

def geometry_fingerprint05(atoms):
    payload = {
        "atomic_numbers": atoms.get_atomic_numbers().astype(int).tolist(),
        "positions_A": np.round(atoms.positions, 8).tolist(),
        "cell_vectors_A": np.round(atoms.cell.array, 8).tolist(),
        "layer_tags": atoms.get_tags().astype(int).tolist(),
        "periodic_xyz": [bool(v) for v in atoms.pbc],
    }
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

structures05 = {}
structure_audit_rows = []
for path in structure_paths:
    candidate_id = path.stem
    atoms = read(path, format="extxyz")
    record = registry05.loc[registry05["candidate_id"].eq(candidate_id)]
    assert len(record) == 1
    record = record.iloc[0]
    assert len(atoms) == 36
    assert atoms.get_chemical_formula() == record["chemical_formula"], candidate_id
    assert atoms.pbc.tolist() == [True, True, False], candidate_id
    assert np.isfinite(atoms.positions).all()
    assert np.isfinite(atoms.cell.array).all()
    assert set(atoms.get_tags().tolist()) == {1, 2, 3, 4}
    digest = geometry_fingerprint05(atoms)
    assert digest == record["geometry_sha256"], candidate_id
    assert atoms.info.get("candidate_id") == candidate_id, candidate_id
    structures05[candidate_id] = atoms
    structure_audit_rows.append({
        "candidate_id": candidate_id,
        "facet": record["facet"],
        "formula": atoms.get_chemical_formula(),
        "sha256_match": True,
        "periodic_xy": True,
    })

structure_audit05 = pd.DataFrame(structure_audit_rows)
assert len(structures05) == 8
assert structure_audit05["sha256_match"].all()
print("INDEPENDENT GEOMETRY PROVENANCE: PASSED (8/8)")
display(structure_audit05)
print("No geometry or source files modified")

INDEPENDENT GEOMETRY PROVENANCE: PASSED (8/8)


,candidate_id,facet,formula,sha256_match,periodic_xy
0,CU-001-100-AG1-SUB,100,AgCu35,True,True
1,CU-001-100-AG2-NN,100,Ag2Cu34,True,True
2,CU-001-100-AG2-SEP,100,Ag2Cu34,True,True
3,CU-001-111-AG1,111,AgCu35,True,True
4,CU-001-111-AG2-NN,111,Ag2Cu34,True,True
5,CU-001-111-REF,111,Cu36,True,True
6,CU-001-AG1,100,AgCu35,True,True
7,CU-001-REF,100,Cu36,True,True


No geometry or source files modified


## 5.3 — Interpretable structure descriptors (no energy labels)

We can extract exact or operationally defined **geometric descriptors** from the atomic structures without training a model. Initial features are deliberately simple:

- Element counts and Ag atomic fraction.
- Number of Ag atoms on the top and second layers.
- Top-layer Ag fraction (nine sites).
- Surface facet and substitution motif from the authoritative registry.
- Minimum Ag–Ag periodic-image separation where two Ag atoms exist.
- Count of Cu–Ag near-neighbor pairs using a documented **2.95 Å geometric cutoff**.
- Mean coordination of top-layer Cu atoms under the same cutoff.

The cutoff is a **fixed computational convention**, not an inferred chemical bond or a learned parameter. Descriptor values can depend on facet, slab construction, cutoff and relaxation. These features are **not catalytic selectivity targets**.

In [3]:
# PSEUDOCODE
# 1. For each validated structure, compute only atomic-geometry descriptors.
# 2. Count Cu–Ag close pairs once, using minimum-image distances and a fixed cutoff.
# 3. Preserve facet, motif, candidate identity, fingerprint, and review status.
# 4. Confirm element counts and top-layer fractions agree with Notebook 04.
# 5. Return an in-memory descriptor table; do not write outputs.

NEIGHBOR_CUTOFF_A = 2.95  # Geometry-only operational threshold; not a fitted bond criterion.
descriptor_rows = []
for candidate_id, atoms in structures05.items():
    rec = registry05.loc[registry05["candidate_id"].eq(candidate_id)].iloc[0]
    symbols = np.asarray(atoms.get_chemical_symbols())
    tags = np.asarray(atoms.get_tags())
    ag = symbols == "Ag"
    cu = symbols == "Cu"
    top = tags == 1
    near = atoms.get_all_distances(mic=True) < NEIGHBOR_CUTOFF_A
    np.fill_diagonal(near, False)
    # Each (Cu index, Ag index) combination contributes only once.
    cu_ag_pairs = int(near[np.ix_(cu, ag)].sum())
    top_cu_counts = near[np.ix_(top & cu, np.ones(len(atoms), dtype=bool))].sum(axis=1)
    mean_top_cu_cn = float(top_cu_counts.mean()) if len(top_cu_counts) else float("nan")
    ag_idx = np.flatnonzero(ag)
    ag_pair_dist = float(atoms.get_distance(int(ag_idx[0]), int(ag_idx[1]), mic=True)) if len(ag_idx) == 2 else float("nan")
    row = {
        "candidate_id": candidate_id,
        "facet": rec["facet"],
        "motif": rec["motif"],
        "chemical_formula": rec["chemical_formula"],
        "n_atoms": len(atoms),
        "n_cu": int(cu.sum()),
        "n_ag": int(ag.sum()),
        "ag_atomic_fraction": float(ag.mean()),
        "ag_top_layer_count": int(np.count_nonzero(ag & top)),
        "ag_second_layer_count": int(np.count_nonzero(ag & (tags == 2))),
        "ag_top_site_fraction": float(np.count_nonzero(ag & top) / np.count_nonzero(top)),
        "ag_ag_min_image_A": ag_pair_dist,
        "cu_ag_near_pair_count": cu_ag_pairs,
        "mean_top_cu_coordination": mean_top_cu_cn,
        "neighbor_cutoff_A": NEIGHBOR_CUTOFF_A,
        "geometry_sha256": geometry_fingerprint05(atoms),
        "structure_review_status": rec["structure_review_status"],
        "approved_for_modeling": False,
        "descriptor_scope": "geometry_only_unrelaxed",
    }
    assert row["n_cu"] + row["n_ag"] == 36
    assert row["n_ag"] == int(rec["ag_count"])
    assert row["ag_top_layer_count"] == int(rec["ag_top_layer_count"])
    assert np.isclose(row["ag_top_site_fraction"], float(rec["ag_top_site_fraction"]), atol=1e-6)
    if len(ag_idx) == 2:
        assert np.isclose(ag_pair_dist, float(rec["ag_ag_minimum_image_distance_A"]), atol=1e-5)
    descriptor_rows.append(row)

features05 = pd.DataFrame(descriptor_rows).sort_values("candidate_id").reset_index(drop=True)
assert len(features05) == 8 and features05["candidate_id"].is_unique
assert not features05["approved_for_modeling"].any()
print("GEOMETRY-ONLY DESCRIPTOR EXTRACTION: PASSED")
display(features05[[
    "candidate_id", "facet", "n_ag", "ag_top_layer_count", "ag_ag_min_image_A",
    "cu_ag_near_pair_count", "mean_top_cu_coordination", "descriptor_scope"
]])
print("No energy or selectivity targets produced; no files modified")

GEOMETRY-ONLY DESCRIPTOR EXTRACTION: PASSED


,candidate_id,facet,n_ag,ag_top_layer_count,ag_ag_min_image_A,cu_ag_near_pair_count,mean_top_cu_coordination,descriptor_scope
0,CU-001-100-AG1-SUB,100,1,0,NaN,12,8.0,geometry_only_unrelaxed
1,CU-001-100-AG2-NN,100,2,2,2.556191,14,8.0,geometry_only_unrelaxed
2,CU-001-100-AG2-SEP,100,2,2,3.615000,16,8.0,geometry_only_unrelaxed
3,CU-001-111-AG1,111,1,1,NaN,9,9.0,geometry_only_unrelaxed
4,CU-001-111-AG2-NN,111,2,2,2.556191,16,9.0,geometry_only_unrelaxed
5,CU-001-111-REF,111,0,0,NaN,0,9.0,geometry_only_unrelaxed
6,CU-001-AG1,100,1,1,NaN,8,8.0,geometry_only_unrelaxed
7,CU-001-REF,100,0,0,NaN,0,8.0,geometry_only_unrelaxed


No energy or selectivity targets produced; no files modified


## 5.4 — Controlled motif comparison and representation checks

A physically meaningful structural comparison must match **composition**, **facet**, **slab size**, and **geometric conventions**. In this initial library, the clearest controlled pair is:

- `CU-001-100-AG2-NN` — Ag₂Cu₃₄, two neighboring Ag atoms on Cu(100).
- `CU-001-100-AG2-SEP` — Ag₂Cu₃₄, two separated Ag atoms on Cu(100).

We can compare their geometric descriptors, but cannot yet conclude which has a lower formation energy or favors ethylene. Comparing raw slab energies across different facets or different stoichiometries would be scientifically ambiguous without consistent surface and elemental chemical-potential references.

In [4]:
# PSEUDOCODE
# 1. Select the nearest-neighbor and separated-pair models with the same facet/formula.
# 2. Check that their geometry fingerprints differ.
# 3. Compare Ag–Ag separations and Cu–Ag close-pair counts.
# 4. State that no stability/selectivity ranking is supported by geometry alone.

pair_ids = ["CU-001-100-AG2-NN", "CU-001-100-AG2-SEP"]
controlled_pairs05 = features05.set_index("candidate_id").loc[pair_ids]
assert controlled_pairs05["facet"].nunique() == 1
assert controlled_pairs05["chemical_formula"].nunique() == 1
assert controlled_pairs05["geometry_sha256"].nunique() == 2
assert controlled_pairs05["ag_ag_min_image_A"].iloc[0] < controlled_pairs05["ag_ag_min_image_A"].iloc[1]
print("MATCHED-COMPOSITION MOTIF COMPARISON: PASSED")
display(controlled_pairs05[[
    "facet", "chemical_formula", "ag_ag_min_image_A", "cu_ag_near_pair_count",
    "mean_top_cu_coordination", "approved_for_modeling"
]])
print("Geometry differs. Stability, adsorption, and selectivity remain UNKNOWN.")

MATCHED-COMPOSITION MOTIF COMPARISON: PASSED


,facet,chemical_formula,ag_ag_min_image_A,cu_ag_near_pair_count,mean_top_cu_coordination,approved_for_modeling
candidate_id,,,,,,
CU-001-100-AG2-NN,100,Ag2Cu34,2.556191,14,8.0,False
CU-001-100-AG2-SEP,100,Ag2Cu34,3.615000,16,8.0,False


Geometry differs. Stability, adsorption, and selectivity remain UNKNOWN.


## 5.5 — Computational baselines: what is and is not justified?

Three distinct layers should not be confused:

| Level | Method | What it provides | What it **does not** prove |
|---|---|---|---|
| A | Geometric descriptors | Composition and local coordination | Adsorption energies or selectivity |
| B | ASE Effective Medium Theory (EMT) | Classical-potential energy/force **diagnostic** for Cu/Ag | DFT accuracy or electrochemical response |
| C | Pretrained MLIP, e.g. MACE | Model-estimated energies/forces | Validated CO₂RR activity, potential effects, or FE |

ASE's included EMT calculator supports Cu and Ag and is intended as a simple demonstration/test calculator. It is a useful reproducible **baseline**, not an authoritative label. Reference: [ASE EMT documentation](https://ase.gitlab.io/ase/ase/calculators/emt.html).

**All energetic diagnostics below are disabled by default.** The entire structural library is pending formal review; no notebook cell may silently change those approval decisions.

## 5.6 — Optional EMT diagnostic (manual opt-in, in memory only)

Set `RUN_EMT_DIAGNOSTIC=True` only when intentionally testing the calculator. This is **method-development experimentation**, not approval of the source structures for production modeling. It evaluates *unrelaxed* structures and does not relax them.

For any meaningful energy difference, restrict comparison to same-facet, same-composition, same-supercell motifs under the **same calculator**. Other rows are informative as software outputs only. Record all energies in **eV** and forces in **eV/Å**.

In [5]:
# PSEUDOCODE
# 1. Default the classical-potential diagnostic to OFF.
# 2. If explicitly enabled, compute energies/forces on copies of unrelaxed structures.
# 3. Label all numerical results as EMT predictions, not DFT or experimental labels.
# 4. Report method and units; do not save or update source records.

RUN_EMT_DIAGNOSTIC = False  # Manual opt-in for local diagnostic calculations only.
emt_results05 = None
if not RUN_EMT_DIAGNOSTIC:
    print("EMT DIAGNOSTIC: DISABLED — no energy or force calculations performed")
else:
    from ase.calculators.emt import EMT
    rows = []
    for candidate_id, initial in structures05.items():
        trial = initial.copy()
        trial.calc = EMT()
        energy_eV = float(trial.get_potential_energy())
        forces_eVA = np.asarray(trial.get_forces(), dtype=float)
        assert np.isfinite(energy_eV) and np.isfinite(forces_eVA).all()
        rows.append({
            "candidate_id": candidate_id,
            "facet": features05.set_index("candidate_id").loc[candidate_id, "facet"],
            "formula": trial.get_chemical_formula(),
            "emt_total_energy_eV": energy_eV,
            "emt_max_force_eV_per_A": float(np.linalg.norm(forces_eVA, axis=1).max()),
            "method": "ASE_EMT_unrelaxed_single_point",
            "selectivity_prediction": "not_available",
            "production_model_approved": False,
        })
    emt_results05 = pd.DataFrame(rows).sort_values("candidate_id")
    assert len(emt_results05) == 8
    print("EMT SINGLE-POINT SOFTWARE DIAGNOSTIC: COMPLETED (8/8)")
    display(emt_results05)
    print("Do not compare unlike stoichiometries/facets as stability rankings")
    print("No input files changed; no human review decisions modified")

EMT DIAGNOSTIC: DISABLED — no energy or force calculations performed


## 5.7 — Conditional same-composition EMT contrast

If EMT was deliberately run in Section 5.6, compute a narrowly defined **unrelaxed** energy contrast between the two Cu(100) Ag₂Cu₃₄ motifs:

\[\Delta E_{\mathrm{SEP-NN}} = E_{\mathrm{EMT}}(\mathrm{SEP}) - E_{\mathrm{EMT}}(\mathrm{NN})\]

This is a model-specific, fixed-composition comparison under the chosen unrelaxed conditions. It is **not** a DFT formation energy, surface segregation energy, adsorption energy, or CO₂RR selectivity prediction.

In [6]:
# PSEUDOCODE
# 1. Check whether the user explicitly enabled EMT.
# 2. Require exactly the two matched Cu(100), Ag2Cu34 configurations.
# 3. Calculate a within-method unrelaxed energy difference only if available.
# 4. Never infer an experimentally preferred catalyst or selectivity.

if emt_results05 is None:
    print("MATCHED EMT COMPARISON: SKIPPED — diagnostic energies were not calculated")
else:
    matched = emt_results05.set_index("candidate_id").loc[pair_ids]
    assert matched["facet"].nunique() == 1
    assert matched["formula"].nunique() == 1
    diff_eV = float(
        matched.loc["CU-001-100-AG2-SEP", "emt_total_energy_eV"]
        - matched.loc["CU-001-100-AG2-NN", "emt_total_energy_eV"]
    )
    print(f"Unrelaxed EMT E(SEP) − E(NN) = {diff_eV:.6f} eV")
    print("Interpretation restricted to matched same-method geometries")
    print("Not a DFT benchmark; not a catalytic performance comparison")

MATCHED EMT COMPARISON: SKIPPED — diagnostic energies were not calculated


## 5.8 — Pretrained interatomic potentials: applicability review

Modern MLIPs predict model-dependent potential-energy surfaces from atom types and geometry. Candidate frameworks for later evaluation include:

| Model family | Training emphasis | Present concern for Cu–Ag CO₂RR |
|---|---|---|
| **MACE** foundation models | Materials trajectories; model-specific checkpoint data | Bulk-trained models do not automatically represent biased electrochemical interfaces or adsorbate chemistry |
| **CHGNet** | Materials Project trajectories, including energies, forces, stress and magnetic moments | General materials training does not constitute validation for Ag–Cu surface reaction pathways |

The most recent MACE model names, datasets and licenses should be checked against [official MACE documentation](https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html) before use. CHGNet provenance: [official documentation](https://chgnet.lbl.gov/).

**Critical limitation:** Neither model predicts Faradaic efficiencies or electrode-potential-dependent CO₂RR reaction selectivity from clean Cu–Ag slabs. No transfer of P005 published *CHO energies to these new geometries is permitted without scientifically matched calculations.

Mac M4 development path: start with CPU, run small single-structure tests, and only consider MPS if a model is documented to support that backend. MPS availability in PyTorch does not guarantee each MLIP's operations are supported.

In [7]:
# PSEUDOCODE
# 1. Inspect optional ML packages without installing or downloading them.
# 2. Check whether PyTorch is importable and the Apple MPS backend is available.
# 3. Record compatibility as preliminary; do not run a pretrained calculator.
# 4. Keep the structure/model approval gate unchanged.

mlip_availability05 = {
    "torch": importlib.util.find_spec("torch") is not None,
    "mace": importlib.util.find_spec("mace") is not None,
    "chgnet": importlib.util.find_spec("chgnet") is not None,
}
mps_available05 = False
if mlip_availability05["torch"]:
    import torch
    mps_available05 = bool(torch.backends.mps.is_available())
    print("PyTorch version:", torch.__version__)
print("Optional MLIP packages:", mlip_availability05)
print("PyTorch MPS available:", mps_available05)
print("MACE model execution: NOT REQUESTED")
print("MLIP domain applicability for CO2RR: NOT ESTABLISHED")
print("No packages installed and no checkpoints downloaded")

Optional MLIP packages: {'torch': False, 'mace': False, 'chgnet': False}
PyTorch MPS available: False
MACE model execution: NOT REQUESTED
MLIP domain applicability for CO2RR: NOT ESTABLISHED
No packages installed and no checkpoints downloaded


## 5.9 — Optional pretrained MACE diagnostic: deliberately locked

This section does **not** run automatically. To experiment with MACE, first document:

1. Installed `mace-torch` version and **exact** pretrained checkpoint (the default MACE model has changed across releases).
2. Checkpoint license and training set/functional provenance.
3. Chosen device (`cpu` first on your M4 Mac) and numerical precision.
4. The explicit scientific purpose: **software interoperability and unrelaxed energy/force diagnostic**, *not* catalytic screening approval.
5. Whether the checkpoint download is permitted (the package may download model weights on first use).

Start with **one pure Cu reference** instead of all eight models. Restrict further expansion until its output is reviewed. Consult [MACE's official ASE-calculator documentation](https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html).

In [8]:
# PSEUDOCODE
# 1. Require explicit opt-in to both running MACE and allowing checkpoint access.
# 2. Never download model weights in the default notebook execution.
# 3. When authorized, calculate a single reference structure using an explicit model.
# 4. Label the energy and forces as an unvalidated pretrained-model diagnostic.
# 5. Do not save scores or modify human review decisions.

RUN_MACE_DIAGNOSTIC = False
ALLOW_CHECKPOINT_DOWNLOAD = False
mace_reference05 = None
if not RUN_MACE_DIAGNOSTIC:
    print("MACE DIAGNOSTIC: DISABLED — no checkpoint loaded or downloaded")
else:
    assert ALLOW_CHECKPOINT_DOWNLOAD, (
        "Set ALLOW_CHECKPOINT_DOWNLOAD=True only after accepting the exact model, "
        "version, license, network use, and scientific limitations."
    )
    assert mlip_availability05["mace"], (
        "mace-torch is not installed; installation must be a separate user action."
    )
    from mace.calculators import mace_mp
    model_name = "small"  # Explicit MACE-MP-0 family choice; inspect installed docs first.
    calc = mace_mp(model=model_name, device="cpu", default_dtype="float32")
    trial = structures05["CU-001-REF"].copy()
    trial.calc = calc
    value = float(trial.get_potential_energy())
    forces = np.asarray(trial.get_forces(), dtype=float)
    assert np.isfinite(value) and np.isfinite(forces).all()
    mace_reference05 = {
        "candidate_id": "CU-001-REF",
        "model_requested": model_name,
        "device": "cpu",
        "energy_eV": value,
        "max_force_eV_per_A": float(np.linalg.norm(forces, axis=1).max()),
        "purpose": "single_point_interoperability_only",
        "validated_against_reference_DFT": False,
        "selectivity_predicted": False,
    }
    print("MACE SINGLE-POINT DIAGNOSTIC — NOT A SCIENTIFIC BENCHMARK")
    print(json.dumps(mace_reference05, indent=2))
    print("No source or approval files modified")

MACE DIAGNOSTIC: DISABLED — no checkpoint loaded or downloaded


## 5.10 — Energy and ML evaluation protocol (future, gated)

**A model prediction is not evidence of accuracy.** A defensible energetic comparison would require **matched independent reference data** (e.g., carefully converged DFT for exactly the structures/conditions of interest), including reference functionals, magnetic/spin approximations where applicable, slab thickness, vacuum, constraints, and convergence settings.

A future validation suite should evaluate:

1. **Energy:** MAE/RMSE against independent reference energies *within a consistent definition*, and appropriate relative-energy metrics.
2. **Force:** Cartesian per-component errors and force-norm errors (eV/Å), including high-force unrelaxed examples.
3. **Geometry:** Whether constrained relaxation is stable and physically reasonable, with bottom layers handled consistently.
4. **Applicability:** bulk-to-surface, dilute Ag, clustered Ag, adsorbates, and electrolyte/potential domain shifts.
5. **Uncertainty:** model disagreement/calibration tested against independent references, not inferred from score spread alone.
6. **Leakage:** split by structural families, parent configurations, and sources; do not randomly split near-identical perturbed geometries.
7. **Electrochemistry:** separate energetics/adsorption hypotheses from FE, potential, solvation, local pH, mass transport and reaction kinetics.

**No supervised training or quantitative benchmark is authorized in this notebook without new, matched and adequately reviewed labels.**

In [9]:
# PSEUDOCODE
# 1. Assert the initial structures are still pending visual review.
# 2. Assert there is no verified energy/selectivity target column for this library.
# 3. Record the absence of an independent DFT benchmark and train/test split.
# 4. Do not synthesize labels or mark any model as validated.

assert registry05["structure_review_status"].eq("pending_visual_review").all()
assert not registry05["approved_for_modeling"].str.lower().eq("true").any()
assert registry05["energy_status"].eq("not_computed").all()
assert registry05["selectivity_status"].eq("not_predicted").all()
assert not any(c in features05.columns for c in [
    "dft_reference_energy_eV", "experimental_fe_percent", "target_selectivity"
])

ml_readiness05 = {
    "structures_reloaded": len(structures05),
    "verified_geometry_fingerprints": int(structure_audit05["sha256_match"].sum()),
    "geometry_descriptors_available": len(features05),
    "approved_for_production_modeling": False,
    "independent_reference_DFT_for_structures": False,
    "experimental_selectivity_labels_for_structures": False,
    "supervised_ML_training_performed": False,
    "quantitative_energy_benchmark_completed": False,
    "MLIP_validated_for_electrochemical_CO2RR": False,
    "publication_or_structure_family_holdout_evaluated": False,
}
print("SCIENTIFIC ML READINESS — CONSERVATIVE GATE")
print(json.dumps(ml_readiness05, indent=2))
print("No synthetic labels generated; no approved status changed")

SCIENTIFIC ML READINESS — CONSERVATIVE GATE
{
  "structures_reloaded": 8,
  "verified_geometry_fingerprints": 8,
  "geometry_descriptors_available": 8,
  "approved_for_production_modeling": false,
  "independent_reference_DFT_for_structures": false,
  "experimental_selectivity_labels_for_structures": false,
  "supervised_ML_training_performed": false,
  "quantitative_energy_benchmark_completed": false,
  "MLIP_validated_for_electrochemical_CO2RR": false,
  "publication_or_structure_family_holdout_evaluated": false
}
No synthetic labels generated; no approved status changed


## 5.11 — Optional export of geometry-only descriptors (separate manual gate)

The descriptor table can be saved without pretending it is a supervised ML dataset. This must be a **new output directory**, never overwriting the original Notebook 04 geometry registry or evidence records.

**Default:** `EXPORT_DESCRIPTOR_ONLY = False`. The export writes an eight-row CSV plus a JSON manifest with source geometry digests and scientific restrictions; it refuses to overwrite existing output. It never exports EMT or MACE energies from exploratory diagnostic cells.

In [10]:
# PSEUDOCODE
# 1. Default the geometry-feature export gate to OFF.
# 2. If enabled, refuse overwriting previous output.
# 3. Export descriptors and provenance to a new isolated directory.
# 4. Keep diagnostic energies out of the feature file.
# 5. Verify round-trip descriptors; do not edit source geometries or reviews.

EXPORT_DESCRIPTOR_ONLY = False  # Manually set True only after reviewing the feature table.
FEATURE_EXPORT_ROOT = ROOT / "data" / "processed" / "05_geometry_descriptors"
if not EXPORT_DESCRIPTOR_ONLY:
    print("DESCRIPTOR EXPORT GATE: CLOSED — no files written")
    print("Optional destination:", FEATURE_EXPORT_ROOT)
else:
    import shutil
    import tempfile
    assert not FEATURE_EXPORT_ROOT.exists(), (
        "Descriptor export already exists. Refusing overwrite; version it separately."
    )
    FEATURE_EXPORT_ROOT.parent.mkdir(parents=True, exist_ok=True)
    staging = Path(tempfile.mkdtemp(
        prefix=".05_descriptor_staging_", dir=str(FEATURE_EXPORT_ROOT.parent)
    ))
    try:
        out_dir = staging / FEATURE_EXPORT_ROOT.name
        out_dir.mkdir()
        csv_path = out_dir / "05_geometry_descriptors.csv"
        features05.to_csv(csv_path, index=False)
        reloaded = pd.read_csv(csv_path)
        assert len(reloaded) == 8
        assert set(reloaded["candidate_id"]) == set(features05["candidate_id"])
        assert not reloaded["approved_for_modeling"].any()
        manifest = {
            "campaign_id": "CU-001",
            "notebook": "05_Scientific_ML_and_Baselines",
            "input_folder": "data/processed/04_structures",
            "candidate_count": 8,
            "fingerprints": dict(zip(features05["candidate_id"], features05["geometry_sha256"])),
            "descriptor_scope": "unrelaxed_geometry_only",
            "neighbor_cutoff_A": NEIGHBOR_CUTOFF_A,
            "human_visual_review": "pending",
            "modeling_approved": False,
            "DFT_reference_labels_included": False,
            "experimental_selectivity_targets_included": False,
            "EMT_or_MACE_energies_included": False,
        }
        (out_dir / "05_descriptor_manifest.json").write_text(
            json.dumps(manifest, indent=2) + "\n", encoding="utf-8"
        )
        out_dir.rename(FEATURE_EXPORT_ROOT)
        print("DESCRIPTOR-ONLY EXPORT COMPLETE:", FEATURE_EXPORT_ROOT)
        print("Source geometry and scientific evidence files unchanged")
    finally:
        shutil.rmtree(staging, ignore_errors=True)

DESCRIPTOR EXPORT GATE: CLOSED — no files written
Optional destination: /Users/nsolebna/202609_Ironhack/Week 8/Project 3/CuSelect-AI/data/processed/05_geometry_descriptors


## 5.12 — Scientific findings and handoff to Notebook 06

### What Notebook 05 establishes

- A deterministic link from 8 exported `.extxyz` structures to the authoritative geometry registry and SHA-256 fingerprints.
- A small **unlabeled**, geometry-only feature table for Cu(100), Cu(111), and Cu–Ag substitution motifs.
- A matched-motif comparison (Ag₂Cu₃₄ Cu(100) nearest versus separated substitution) that does not misuse unlike compositions or facets.
- An explicit classical-potential EMT demonstration route and an independent, optional pretrained-MACE diagnostic route — both disabled by default and not equivalent to scientific validation.
- Clear readiness restrictions: **no DFT labels, no catalyst-selectivity labels, no supervised training, and no claim of model accuracy**.

### What must happen before meaningful energy ranking or supervised ML

1. Complete and record a formal human visual review of all eight geometries, including side views and subsurface placement.
2. Build or obtain **matched** independent reference-energy and force data for these atomic structures or a well-characterized broader corpus.
3. Audit reference calculation methods, labels, and potential train/test leakage.
4. Validate MLIP prediction error and applicability to Cu–Ag **surfaces** before screening.
5. For actual CO₂RR selectivity questions, develop interface-/potential-/adsorbate-aware evidence and suitable experimental validation.

### Original roadmap continues

**Next planned notebook:** `06_RAG_and_Evidence_Quality.ipynb` — source-grounded retrieval, citation checking, and scientific abstention. The RAG corpus should use reviewed literature records, not treat experimental FE as a property calculated from clean slabs.

### References and current documentation

- [ASE EMT calculator](https://ase.gitlab.io/ase/ase/calculators/emt.html)
- [ASE constrained structure optimization](https://ase.gitlab.io/ase/ase/optimize.html)
- [MACE foundation-model documentation](https://mace-docs.readthedocs.io/en/latest/guide/foundation_models.html)
- [CHGNet documentation](https://chgnet.lbl.gov/)
- [PyTorch MPS backend documentation](https://docs.pytorch.org/docs/stable/notes/mps.html)

**Interpretation:** This notebook develops an auditable computational-ML baseline. Without independent reference labels, computed model energies remain exploratory estimates, not validated catalyst rankings.

In [11]:
# PSEUDOCODE
# 1. Summarize what was demonstrated and which gates were deliberately left closed.
# 2. Separate geometry-feature readiness from scientific ML validation.
# 3. Confirm no source approval statuses were modified.
# 4. Name the next notebook in the original CuSelect-AI plan.

summary05 = {
    "campaign_id": "CU-001",
    "notebook": "05_Scientific_ML_and_Baselines",
    "source_structures": len(structures05),
    "geometry_fingerprint_checks_passed": int(structure_audit05["sha256_match"].sum()),
    "geometry_only_feature_rows": len(features05),
    "matched_facet_and_composition_pair_count": 1,
    "emt_diagnostic_ran": emt_results05 is not None,
    "mace_diagnostic_ran": mace_reference05 is not None,
    "geometry_descriptor_export_exists": FEATURE_EXPORT_ROOT.exists(),
    "human_visual_review": "pending",
    "modeling_approved": False,
    "reference_DFT_benchmark_completed": False,
    "selectivity_predicted": False,
    "next_notebook": "06_RAG_and_Evidence_Quality",
}
assert summary05["source_structures"] == 8
assert summary05["geometry_fingerprint_checks_passed"] == 8
assert summary05["modeling_approved"] is False
print("NOTEBOOK 05 — SCIENTIFIC ML & BASELINES SUMMARY")
print(json.dumps(summary05, indent=2))
print("\nNo source geometry, evidence, or human-review record modified")

NOTEBOOK 05 — SCIENTIFIC ML & BASELINES SUMMARY
{
  "campaign_id": "CU-001",
  "notebook": "05_Scientific_ML_and_Baselines",
  "source_structures": 8,
  "geometry_fingerprint_checks_passed": 8,
  "geometry_only_feature_rows": 8,
  "matched_facet_and_composition_pair_count": 1,
  "emt_diagnostic_ran": false,
  "mace_diagnostic_ran": false,
  "geometry_descriptor_export_exists": true,
  "human_visual_review": "pending",
  "modeling_approved": false,
  "reference_DFT_benchmark_completed": false,
  "selectivity_predicted": false,
  "next_notebook": "06_RAG_and_Evidence_Quality"
}

No source geometry, evidence, or human-review record modified
